<a href="https://colab.research.google.com/github/keerthisrimandapati01-art/ML-Practical/blob/main/exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os
os.listdir('/content/drive/MyDrive/colab')

['placement_predict_50k.csv',
 'placement_dataset.csv',
 'placement_predict_50k_adjusted.csv']

In [15]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/colab/placement_predict_50k_adjusted.csv')

In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------

# Read the dataset from Google Drive
df = pd.read_csv("/content/drive/MyDrive/colab/placement_predict_50k.csv")

# Target column
target_col = "CGPA"

# Drop unnecessary columns
drop_cols = [target_col, "StudentID", "CGPA_Tier"]

feature_df = df.drop(columns=drop_cols).select_dtypes(include=[np.number])

# Fill missing values with median
feature_df = feature_df.fillna(feature_df.median())

# Features and target
X = feature_df.values
y = df[target_col].values.reshape(-1, 1)

# ---------------------------------------------------------
# 2. Train/Test Split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------------
# 3. Feature Scaling
# ---------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Add bias column
X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# ---------------------------------------------------------
# 4. Gradient Descent Function
# ---------------------------------------------------------

def gradient_descent(X, y, lr=0.01, n_iters=1000):

    m, n = X.shape

    theta = np.zeros((n, 1))
    losses = []

    for i in range(n_iters):

        y_pred = X @ theta

        error = y_pred - y

        loss = (1 / (2 * m)) * np.sum(error ** 2)
        losses.append(loss)

        gradient = (1 / m) * (X.T @ error)

        theta = theta - lr * gradient

        if i % 100 == 0:
            print(f"Iteration {i:4d} | Loss = {loss:.4f}")

    return theta, losses

# Train model
theta, losses = gradient_descent(
    X_train_b,
    y_train,
    lr=0.01,
    n_iters=1000
)

# ---------------------------------------------------------
# 5. Evaluation
# ---------------------------------------------------------

y_pred = X_test_b @ theta

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nFinal Weights (Theta):")
print(theta.ravel())

print("\nMean Squared Error :", mse)
print("R2 Score :", r2)

# ---------------------------------------------------------
# 6. Compare with Scikit-Learn
# ---------------------------------------------------------

sk_model = LinearRegression()

sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)

print("\nScikit-Learn Linear Regression")
print("R2 Score :", r2_score(y_test, sk_pred))
print("MSE :", mean_squared_error(y_test, sk_pred))

Iteration    0 | Loss = 27.1394
Iteration  100 | Loss = 3.5738
Iteration  200 | Loss = 0.4812
Iteration  300 | Loss = 0.0661
Iteration  400 | Loss = 0.0100
Iteration  500 | Loss = 0.0021
Iteration  600 | Loss = 0.0008
Iteration  700 | Loss = 0.0005
Iteration  800 | Loss = 0.0003
Iteration  900 | Loss = 0.0002

Final Weights (Theta):
[ 7.29885264e+00  1.21681011e-01  1.20862482e-01  1.20929025e-01
  1.20521930e-01  1.20939122e-01  1.20537979e-01  1.22192524e-01
  1.20884881e-01  1.46046278e-02  2.77001333e-03  5.93393027e-03
  2.70514399e-03  5.98236623e-03  6.42601090e-04  1.30829766e-02
  1.49056735e-02  1.65135353e-02  1.21536858e-02 -8.40508047e-05
  1.52383243e-05]

Mean Squared Error : 0.00034511147484136035
R2 Score : 0.9996615850411459

Scikit-Learn Linear Regression
R2 Score : 0.9999914423330684
MSE : 8.72700505316537e-06
